# 🚦 Báo cáo Pipeline Làm Sạch Dữ Liệu Giao Thông

Notebook này trình bày quy trình **làm sạch dữ liệu giao thông (Traffic Data Cleaning Pipeline)** 
được xây dựng bằng Python.

Pipeline thực hiện các bước:

1. Đọc dữ liệu thô từ **MinIO Object Storage**
2. Nạp bảng **locations**
3. Làm sạch dữ liệu theo các quy tắc
4. Lưu dữ liệu hợp lệ vào **SQLite Clean Database**
5. Đảm bảo **không tạo dữ liệu trùng lặp**

---

## Kiến trúc pipeline

Raw Data (MinIO)  
⬇  
Parquet Files  
⬇  
Python Cleaning Script  
⬇  
Data Validation  
⬇  
SQLite Clean Database

In [5]:
import sqlite3
from datetime import datetime
import os
import io
import pandas as pd
from minio import Minio

## 3. Cấu hình hệ thống

Thiết lập các tham số kết nối đến MinIO và đường dẫn lưu dữ liệu sạch.

- MINIO_ENDPOINT: địa chỉ server MinIO
- BUCKET_NAME: bucket chứa dữ liệu giao thông
- PREFIX: thư mục chứa dữ liệu traffic
- CLEAN_DB_PATH: đường dẫn database lưu dữ liệu đã clean

In [6]:

MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin123"

BUCKET_NAME = "raw-traffic-data"
PREFIX = "traffic/incremental"
LOCATIONS_PREFIX = "traffic/locations"

CLEAN_DIR = "data/clean"
CLEAN_DB_PATH = os.path.join(CLEAN_DIR, "data_traffic_clean.db")

os.makedirs(CLEAN_DIR, exist_ok=True)

print("Database path:", CLEAN_DB_PATH)

Database path: data/clean\data_traffic_clean.db


## 4. Kết nối MinIO

MinIO là hệ thống object storage dùng để lưu trữ dữ liệu thô.
Trong bước này, hệ thống tạo client để truy cập và đọc các file Parquet từ bucket.

In [7]:
# ================= MINIO CLIENT =================

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

print("Kết nối MinIO thành công")

Kết nối MinIO thành công


## 5. Tạo database SQLite

Dữ liệu sau khi làm sạch sẽ được lưu vào SQLite.

Hai bảng chính:

1. traffic_data_clean
   - lưu dữ liệu giao thông đã clean

2. locations
   - lưu thông tin vị trí các điểm giao thông

In [8]:
# ================= SQLITE DATABASE =================

conn = sqlite3.connect(CLEAN_DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS traffic_data_clean (
    id INTEGER,
    timestamp TEXT,
    location TEXT,
    current_speed_kmh REAL,
    free_flow_speed_kmh REAL,
    speed_ratio REAL,
    traffic_level TEXT,
    confidence REAL,
    PRIMARY KEY (id, timestamp)
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS locations (
    id INTEGER PRIMARY KEY,
    name TEXT,
    latitude REAL,
    longitude REAL
)
""")

conn.commit()

print("Đã tạo bảng trong SQLite")

Đã tạo bảng trong SQLite


## 6. Tải bảng locations từ MinIO

Bảng locations chứa thông tin về các điểm giao thông như:

- tên địa điểm
- tọa độ

Hệ thống đọc file Parquet mới nhất từ MinIO và lưu vào SQLite.

In [9]:
print("Đang load bảng locations từ MinIO...")

loc_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix=LOCATIONS_PREFIX,
        recursive=True
    )
)

loc_files = sorted(
    [obj.object_name for obj in loc_objects if obj.object_name.endswith(".parquet")]
)

if len(loc_files) > 0:

    latest_loc = loc_files[-1]

    response = client.get_object(BUCKET_NAME, latest_loc)
    df_locations = pd.read_parquet(io.BytesIO(response.read()))
    response.close()

    df_locations.to_sql("locations", conn, if_exists="replace", index=False)

    print("Đã load bảng locations:", latest_loc)

    df_locations.head()

else:
    print("Không tìm thấy file locations")

Đang load bảng locations từ MinIO...
Đã load bảng locations: traffic/locations/locations_20260310_144741.parquet


In [10]:
df = pd.read_sql_query("SELECT * FROM locations LIMIT 10", conn)
df

,id,name,lat,lon,active
0,1,NGÃ 5 ĐỐNG ĐA,13.783255,109.219690,1
1,2,HOÀNG VĂN THỤ - TÂY SƠN,13.759429,109.205798,1
2,3,VÒNG XOAY QUẢNG TRƯỜNG NTT,13.771845,109.222182,1
3,4,VÒNG XOAY NGUYỄN THÁI HỌC,13.775568,109.222460,1
4,5,NGÃ 3 THÁP ĐÔI,13.785601,109.210376,1
5,6,NGUYỄN THỊ ĐỊNH - TRẦN QUANG KHẢI,13.752202,109.210807,1
6,7,NGUYỄN THÁI HỌC - LÊ DUẨN,13.774639,109.221278,1
7,8,TĂNG BẠT HỔ - PHAN CHU TRINH,13.772899,109.237273,1
8,9,PHỐ ẨM THỰC PHAN BỘI CHÂU,13.773845,109.234940,1
9,10,HAI BÀ TRƯNG - LÊ THÁNH TÔNG,13.771955,109.234914,1


## 7. Tìm các file dữ liệu giao thông

Dữ liệu giao thông được lưu dưới dạng nhiều file Parquet.
Hệ thống sẽ tìm tất cả các file trong thư mục traffic/incremental.

In [11]:
print("Đang tìm file parquet trong MinIO...")

objects = client.list_objects(
    BUCKET_NAME,
    prefix=PREFIX,
    recursive=True
)

parquet_files = [
    obj.object_name
    for obj in objects
    if obj.object_name.endswith(".parquet")
]

print("Số file parquet tìm được:", len(parquet_files))

parquet_files[:5]

Đang tìm file parquet trong MinIO...
Số file parquet tìm được: 1


['traffic/incremental/2026-03-10/traffic_20260310_144741.parquet']

In [12]:
sample_file = parquet_files[0]

response = client.get_object(BUCKET_NAME, sample_file)
df_raw = pd.read_parquet(io.BytesIO(response.read()))
response.close()

df_raw.head()

,id,timestamp,location,current_speed_kmh,free_flow_speed_kmh,speed_ratio,traffic_level,confidence
0,1,2026-01-12 20:19:56,1,53.0,53.0,1.00,THOANG,1.000000
1,2,2026-01-12 20:19:56,2,39.0,49.0,0.80,DONG,0.990786
2,3,2026-01-12 20:19:56,3,29.0,38.0,0.76,DONG,0.914126
3,4,2026-01-12 20:19:56,4,29.0,39.0,0.74,DONG,0.940000
4,5,2026-01-12 20:19:56,5,30.0,39.0,0.77,DONG,0.928320


## 8. Làm sạch dữ liệu

Quá trình làm sạch bao gồm:

1. Loại bỏ bản ghi có giá trị NULL
2. Loại bỏ dữ liệu có free_flow_speed_kmh ≤ 0
3. Loại bỏ dữ liệu có current_speed_kmh < 0
4. Loại bỏ bản ghi có confidence <= 0.9


In [ ]:
print("Bắt đầu quá trình cleaning...")

for obj_name in parquet_files:

    response = client.get_object(BUCKET_NAME, obj_name)
    df_raw = pd.read_parquet(io.BytesIO(response.read()))
    response.close()

    for _, row in df_raw.iterrows():

        rid = row["id"]
        ts = row["timestamp"]
        location = row["location"]

        current_speed = row["current_speed_kmh"]
        free_flow_speed = row["free_flow_speed_kmh"]

        speed_ratio = row["speed_ratio"]
        traffic_level = row["traffic_level"]

        confidence = row.get("confidence", None)

        # validation rules

        if pd.isna(current_speed) or pd.isna(free_flow_speed):
            continue

        if free_flow_speed <= 0 or current_speed < 0:
            continue

        if pd.isna(speed_ratio) or pd.isna(traffic_level):
            continue

        if pd.isna(confidence) or confidence <= 0.9:
            continue

        cur.execute("""
        INSERT OR IGNORE INTO traffic_data_clean
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            rid,
            ts,
            location,
            current_speed,
            free_flow_speed,
            speed_ratio,
            traffic_level,
            confidence
        ))

conn.commit()

print("Cleaning hoàn tất")

Bắt đầu quá trình cleaning...
Cleaning hoàn tất


In [14]:
df_clean = pd.read_sql_query(
"""
SELECT *
FROM traffic_data_clean
LIMIT 20
""",
conn
)

df_clean

,id,timestamp,location,current_speed_kmh,free_flow_speed_kmh,speed_ratio,traffic_level,confidence
0,1,2026-01-12 20:19:56,1,53.0,53.0,1.00,THOANG,1.000000
1,2,2026-01-12 20:19:56,2,39.0,49.0,0.80,DONG,0.990786
2,3,2026-01-12 20:19:56,3,29.0,38.0,0.76,DONG,0.914126
3,4,2026-01-12 20:19:56,4,29.0,39.0,0.74,DONG,0.940000
4,5,2026-01-12 20:19:56,5,30.0,39.0,0.77,DONG,0.928320
5,6,2026-01-12 20:19:56,6,38.0,49.0,0.78,DONG,0.990786
6,7,2026-01-12 20:19:56,7,19.0,37.0,0.51,DONG,0.990000
7,8,2026-01-12 20:19:56,8,21.0,34.0,0.62,DONG,0.961674
8,9,2026-01-12 20:19:56,9,30.0,40.0,0.75,DONG,0.928858
9,10,2026-01-12 20:19:56,10,21.0,34.0,0.62,DONG,0.961674


In [16]:
# =====================
# Đếm tổng dữ liệu RAW
# =====================

total_raw = 0

for obj_name in parquet_files:

    response = client.get_object(BUCKET_NAME, obj_name)
    df = pd.read_parquet(io.BytesIO(response.read()))
    response.close()

    total_raw += len(df)

print("Tổng số bản ghi RAW:", total_raw)


# =====================
# Đếm dữ liệu CLEAN
# =====================

clean_count = pd.read_sql_query(
"""
SELECT COUNT(*) as total
FROM traffic_data_clean
""",
conn
).iloc[0]["total"]

print("Tổng số bản ghi CLEAN:", clean_count)


# =====================
# Tính số bản ghi bị loại
# =====================

removed = total_raw - clean_count
removed_ratio = removed / total_raw * 100


# =====================
# Tạo bảng so sánh
# =====================

summary = pd.DataFrame({
    "Dataset": ["Raw Data", "Clean Data", "Removed"],
    "Records": [total_raw, clean_count, removed]
})

summary

Tổng số bản ghi RAW: 29375
Tổng số bản ghi CLEAN: 27421


,Dataset,Records
0,Raw Data,29375
1,Clean Data,27421
2,Removed,1954


# Kết luận

Pipeline làm sạch dữ liệu đảm bảo:

- Loại bỏ dữ liệu thiếu
- Loại bỏ dữ liệu không hợp lệ
- Chỉ giữ dữ liệu có độ tin cậy cao
- Không tạo dữ liệu trùng lặp

Database sau khi làm sạch có thể sử dụng cho:

- Phân tích giao thông
- Dashboard
- Machine Learning
- Dự báo ùn tắc